# <font color="steelblue">Despliegue de modelos con Gradio</font>

**Material desarrollado por los [equipos de trabajo de IA4LEGOS](https://ia4legos.umh.es/)**


**Fecha última edición**: 10/06/2026

**Licencia**: <small><a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a><br /></small>

No olvides hacer una copia si deseas utilizarlo. Al usar estos contenidos, aceptas nuestros términos de uso y nuestra política de privacidad.

---

# Índice

| Módulo | Contenido |
|---|---|
| 0 | El modelo mental: qué es y qué no es Gradio |
| 1 | Instalación y primera aplicación |
| 2 | `gr.Interface`: la vía rápida |
| 3 | Catálogo de componentes |
| 4 | `gr.Blocks`: control total del layout |
| 5 | El sistema de eventos |
| 6 | Estado, validación y funciones avanzadas |
| 7 | Del cuaderno a la aplicación: refactorización |
| 8 | **Despliegue permanente en Hugging Face Spaces** |
| 9 | Alternativas de despliegue |
| 10 | Buenas prácticas y errores frecuentes |
| A | Apéndice: cambios de Gradio 5 a Gradio 6 |
| B | Apéndice: chuleta de referencia rápida |

---

# Módulo 0. El modelo mental

## 0.1 La idea central

Gradio se basa en una única abstracción que conviene interiorizar antes de escribir nada:

> **Una aplicación de Gradio es una función de Python con una interfaz web pegada delante.**

Tú escribes una función normal y corriente:

```python
def predecir(medida_1, medida_2):
    return modelo.predict([[medida_1, medida_2]])[0]
```

Y Gradio se encarga de:

1. Dibujar los controles de entrada (dos cajas de texto, dos sliders...).
2. Recoger lo que el usuario introduce.
3. Llamar a tu función con esos valores.
4. Coger lo que la función devuelve y pintarlo en pantalla.

Todo lo demás (HTML, CSS, JavaScript, servidor web, rutas, peticiones HTTP) queda oculto. Esa es la razón por la que se puede tener una demo funcionando en cinco líneas.

## 0.2 Consecuencias prácticas de ese modelo

Este diseño tiene implicaciones que explican casi todo el comportamiento de Gradio:

**El número de entradas de la interfaz debe coincidir con el número de argumentos de la función.** Si declaras tres componentes de entrada, tu función recibe tres argumentos posicionales, en ese orden.

**El número de salidas debe coincidir con lo que la función devuelve.** Si declaras tres componentes de salida, tu función debe devolver una tupla de tres elementos.

**Cada componente tiene un "tipo de dato Python" asociado.** Un `gr.Slider` entrega un `float`. Un `gr.Image` entrega un `numpy.ndarray` (o una ruta, según se configure). Un `gr.Dataframe` entrega un `pandas.DataFrame`. Conocer esta correspondencia es el 80 % de la depuración en Gradio.

**La función se ejecuta en el servidor.** El usuario ve una web en su navegador, pero el código Python corre en tu máquina o en el servidor donde despliegues. Esto es lo que permite usar modelos de scikit-learn o PyTorch sin convertirlos a JavaScript.

## 0.3 Cuándo Gradio y cuándo no

Merece la pena ser honesto sobre los límites, porque elegir mal la herramienta cuesta tiempo:

**Gradio encaja bien cuando:**
- Quieres demostrar un modelo: entra un dato, sale una predicción.
- Necesitas un enlace compartible en minutos, no en días.
- El público objetivo son colegas, clientes o revisores, no miles de usuarios concurrentes.
- Trabajas con imágenes, audio, texto o vídeo (los componentes multimedia son excelentes).

**Gradio encaja mal cuando:**
- Necesitas una aplicación con navegación entre páginas, sesiones de usuario, base de datos y lógica de negocio compleja. Eso es una aplicación web: usa FastAPI + un frontend real.
- Necesitas control milimétrico del diseño visual. Gradio te da temas y algo de CSS, pero no es un framework de UI.
- Necesitas un dashboard analítico con muchos filtros cruzados. Ahí Streamlit o Dash suelen ser mejores, como bien indica la tabla comparativa del cuaderno original.

Un criterio simple: **si el centro de tu aplicación es un modelo, usa Gradio; si el centro son los datos, usa Streamlit o Dash.**

---

# Módulo 1. Instalación y primera aplicación

## 1.1 Instalación

```bash
pip install gradio
```

Gradio requiere **Python 3.10 o superior**. En Colab basta con la misma línea precedida de `!`:

```python
!pip install gradio --quiet
```

Comprobación:

```python
import gradio as gr
print(gr.__version__)
```

> **Aviso de versiones.** Gradio 6 introdujo cambios que rompen código escrito para Gradio 5 (el apéndice A los detalla). Si sigues un tutorial antiguo y algo no funciona, lo primero que hay que mirar es la versión. Para trabajo serio, **fija la versión** en `requirements.txt` en lugar de dejar que pip instale la última.

## 1.2 La aplicación mínima

```python
import gradio as gr

def saludar(nombre):
    return f"Hola, {nombre}"

demo = gr.Interface(fn=saludar, inputs="text", outputs="text")
demo.launch()
```

Tres líneas de lógica. Al ejecutarlo verás:

```
Running on local URL:  http://127.0.0.1:7860
```

En Colab, además, Gradio detecta el entorno y muestra la interfaz incrustada en la celda de salida.


## 1.3 Anatomía de lo que acaba de pasar

Vale la pena desmenuzarlo porque este patrón se repite siempre:

| Elemento | Papel |
|---|---|
| `saludar` | Tu lógica. Una función Python cualquiera. |
| `fn=saludar` | Le dices a Gradio qué función ejecutar. |
| `inputs="text"` | Un componente de entrada. `"text"` es un atajo de `gr.Textbox()`. |
| `outputs="text"` | Un componente de salida. |
| `demo.launch()` | Arranca el servidor web. |

Los atajos de cadena (`"text"`, `"number"`, `"image"`, `"slider"`, `"checkbox"`...) son cómodos para prototipar, pero **en cuanto quieras configurar algo** (etiquetas, rangos, valores por defecto) tendrás que usar la clase completa:

```python
inputs=gr.Textbox(label="Tu nombre", placeholder="Escribe aquí...")
```

---

# Módulo 2. `gr.Interface`: la vía rápida

`gr.Interface` genera automáticamente un layout de dos columnas (entradas a la izquierda, salidas a la derecha) con botones de enviar y limpiar. Es la forma más rápida de envolver una función.

## 2.1 Firma y parámetros importantes

```python
gr.Interface(
    fn,                      # la función a ejecutar
    inputs,                  # componente o lista de componentes
    outputs,                 # componente o lista de componentes
    title=None,              # título en grande
    description=None,        # texto bajo el título (acepta Markdown)
    article=None,            # texto largo al pie de la página
    examples=None,           # lista de listas con valores de ejemplo
    cache_examples=False,    # precalcula los ejemplos al arrancar
    live=False,              # ejecuta al cambiar cualquier entrada, sin botón
    flagging_mode="never",   # desactiva el botón de "marcar" (recomendado)
    submit_btn="Enviar",     # texto del botón
    clear_btn="Limpiar",
)
```

## 2.2 Ejemplo con múltiples entradas y salidas

```python
import gradio as gr

def calcular_imc(peso, altura, sexo):
    imc = peso / (altura ** 2)
    if imc < 18.5:
        categoria = "Bajo peso"
    elif imc < 25:
        categoria = "Peso normal"
    elif imc < 30:
        categoria = "Sobrepeso"
    else:
        categoria = "Obesidad"
    detalle = f"IMC de {imc:.1f} para un {sexo.lower()} de {peso} kg y {altura} m"
    return round(imc, 2), categoria, detalle

demo = gr.Interface(
    fn=calcular_imc,
    inputs=[
        gr.Number(label="Peso (kg)", value=70),
        gr.Number(label="Altura (m)", value=1.75),
        gr.Radio(["Hombre", "Mujer"], label="Sexo", value="Hombre"),
    ],
    outputs=[
        gr.Number(label="IMC"),
        gr.Textbox(label="Categoría"),
        gr.Textbox(label="Detalle"),
    ],
    title="Calculadora de IMC",
    description="Introduce tus datos para calcular el índice de masa corporal.",
    examples=[
        [70, 1.75, "Hombre"],
        [58, 1.62, "Mujer"],
    ],
    flagging_mode="never",
)

demo.launch()
```

Fíjate en el orden: `inputs` tiene tres elementos, la función tiene tres parámetros, `outputs` tiene tres elementos y la función devuelve una tupla de tres. **Esa correspondencia es obligatoria.**

## 2.3 El parámetro `live`

Con `live=True` desaparece el botón de enviar y la función se ejecuta cada vez que cambia cualquier entrada:

```python
demo = gr.Interface(fn=calcular_imc, inputs=[...], outputs=[...], live=True)
```

Útil para funciones rápidas (un cálculo, una predicción con un modelo pequeño). **Contraproducente** para funciones lentas: cada tecla pulsada dispararía una ejecución.

## 2.4 Cuándo abandonar `gr.Interface`

`gr.Interface` es un molde rígido. En cuanto necesites:

- Colocar los componentes en un orden distinto al de dos columnas.
- Que un botón actualice solo una parte de la pantalla.
- Pestañas, acordeones, o componentes que aparecen y desaparecen.
- Más de un botón con acciones diferentes.

...necesitas `gr.Blocks` (módulo 4). No es más difícil, solo más explícito.

---

# Módulo 3. Catálogo de componentes

Este es el vocabulario del framework. No hace falta memorizarlo, pero sí saber qué existe y, sobre todo, **qué tipo de dato Python entrega o espera cada uno**.

## 3.1 Entrada de datos simples

| Componente | Tipo Python | Parámetros clave |
|---|---|---|
| `gr.Textbox()` | `str` | `lines`, `placeholder`, `max_lines`, `type="password"` |
| `gr.Number()` | `float` / `int` | `value`, `minimum`, `maximum`, `precision` |
| `gr.Slider()` | `float` | `minimum`, `maximum`, `step`, `value` |
| `gr.Checkbox()` | `bool` | `value` |
| `gr.CheckboxGroup()` | `list[str]` | `choices`, `value` |
| `gr.Radio()` | `str` | `choices`, `value` |
| `gr.Dropdown()` | `str` o `list[str]` | `choices`, `multiselect`, `allow_custom_value` |
| `gr.ColorPicker()` | `str` (hex) | `value` |
| `gr.DateTime()` | `str` / timestamp | `include_time`, `type` |

Ejemplo con los más habituales en ML:

```python
gr.Slider(minimum=0, maximum=10, step=0.1, value=5.0,
          label="Concentración", info="Valor en mg/L")
```

El parámetro `info` muestra una ayuda pequeña bajo la etiqueta. Es la forma más barata de hacer una interfaz comprensible; úsalo.

## 3.2 Entrada de ficheros y multimedia

| Componente | Tipo Python | Notas |
|---|---|---|
| `gr.File()` | objeto con `.name` (ruta) | `file_types=[".csv"]`, `file_count="multiple"` |
| `gr.Image()` | `numpy.ndarray` por defecto | `type="pil"`, `type="filepath"` |
| `gr.Audio()` | `(sample_rate, ndarray)` | `type="filepath"`, `sources=["microphone"]` |
| `gr.Video()` | ruta `str` | `sources=["webcam"]` |
| `gr.Dataframe()` | `pandas.DataFrame` | `headers`, `datatype`, `interactive` |

**Trampa frecuente con `gr.File`.** El objeto que llega a tu función no es el contenido del fichero, sino un objeto temporal. Para leerlo hay que acceder a su ruta:

```python
def procesar(archivo):
    if archivo is None:
        raise gr.Error("Sube un fichero primero")
    df = pd.read_csv(archivo.name)   # <- .name, no el objeto directamente
    return df.describe()
```

Esto es exactamente lo que hace la función `analizar_csv` del cuaderno original.

**Trampa frecuente con `gr.Image`.** Por defecto entrega un array de NumPy con forma `(alto, ancho, 3)` en RGB. Si tu modelo espera un objeto PIL, declara `gr.Image(type="pil")`. Si espera una ruta en disco, `gr.Image(type="filepath")`. Elegir bien aquí evita una conversión manual.

## 3.3 Salida de resultados

| Componente | Tipo Python que espera | Uso típico |
|---|---|---|
| `gr.Label()` | `dict[str, float]` | **Probabilidades de clasificación** |
| `gr.Textbox()` | `str` | Texto plano |
| `gr.Markdown()` | `str` con Markdown | Informes formateados |
| `gr.JSON()` | `dict` / `list` | Salidas estructuradas |
| `gr.Plot()` | figura matplotlib / plotly | Gráficos |
| `gr.Dataframe()` | `pandas.DataFrame` | Tablas |
| `gr.Gallery()` | lista de imágenes | Varias imágenes a la vez |
| `gr.HTML()` | `str` con HTML | Formato personalizado |
| `gr.File()` | ruta `str` | Descargar un resultado |

**`gr.Label` merece atención especial** porque es el componente estrella para clasificación. Le pasas un diccionario `{clase: probabilidad}` y dibuja automáticamente barras ordenadas:

```python
def clasificar(x):
    probs = modelo.predict_proba([x])[0]
    return {clase: float(p) for clase, p in zip(nombres_clases, probs)}

gr.Label(num_top_classes=3, label="Predicción")
```

El `float(p)` no es decorativo: si le pasas un `numpy.float32` puede fallar la serialización a JSON. Convierte siempre a tipos nativos de Python.

## 3.4 Componentes estructurales y de texto

| Componente | Uso |
|---|---|
| `gr.Markdown("texto")` | Títulos, explicaciones, instrucciones |
| `gr.HTML("<div>...</div>")` | HTML arbitrario |
| `gr.Button("Texto")` | Disparar acciones. `variant="primary"` para destacar |
| `gr.Examples(...)` | Botones con valores predefinidos |
| `gr.State()` | Memoria entre interacciones (módulo 6) |

## 3.5 Parámetros comunes a casi todos

Estos funcionan en la mayoría de componentes y conviene conocerlos:

```python
gr.Textbox(
    label="Etiqueta visible",
    info="Texto de ayuda pequeño",
    value="valor inicial",
    visible=True,        # si False, no se muestra (se puede cambiar en runtime)
    interactive=True,    # si False, solo lectura
    scale=2,             # peso relativo del ancho dentro de una fila
    min_width=160,
    elem_id="mi-id",     # para apuntarlo desde CSS
)
```

`visible` e `interactive` son la base de las interfaces dinámicas: un evento puede cambiarlos para mostrar u ocultar partes de la aplicación.

---

# Módulo 4. `gr.Blocks`: control total del layout

## 4.1 El cambio de mentalidad

Con `gr.Interface` describes *qué* hace la aplicación y Gradio decide *cómo* se ve. Con `gr.Blocks` describes ambas cosas. La estructura general es siempre esta:

```python
with gr.Blocks() as demo:
    # 1. Declaras los componentes (y su posición)
    entrada = gr.Textbox(label="Entrada")
    boton   = gr.Button("Procesar")
    salida  = gr.Textbox(label="Salida")

    # 2. Conectas eventos: qué función se ejecuta, con qué entradas,
    #    y a qué salidas va el resultado
    boton.click(fn=mi_funcion, inputs=entrada, outputs=salida)

demo.launch()
```

Tres ideas nuevas respecto a `gr.Interface`:

1. Los componentes se **declaran dentro de un bloque `with`**. El orden de declaración es el orden en pantalla.
2. Los componentes se **guardan en variables**, para poder referenciarlos después.
3. Las conexiones son **explícitas**: cada evento dice exactamente qué función, qué entradas y qué salidas.

## 4.2 Contenedores de layout

### Filas y columnas

```python
with gr.Blocks() as demo:
    with gr.Row():                    # coloca en horizontal
        with gr.Column(scale=1):      # columna estrecha
            a = gr.Slider(label="A")
            b = gr.Slider(label="B")
        with gr.Column(scale=2):      # columna el doble de ancha
            resultado = gr.Plot()
```

`scale` reparte el ancho proporcionalmente. `scale=1` y `scale=2` significa un tercio y dos tercios.

### Pestañas

```python
with gr.Blocks() as demo:
    with gr.Tab("Predicción"):
        gr.Markdown("Contenido de la primera pestaña")
    with gr.Tab("Explicación del modelo"):
        gr.Markdown("Contenido de la segunda")
    with gr.Tab("Ayuda"):
        gr.Markdown("Instrucciones de uso")
```

Excelente para separar "usar el modelo" de "entender el modelo".

### Acordeón

```python
with gr.Accordion("Opciones avanzadas", open=False):
    semilla = gr.Number(label="Random seed", value=42)
    n_arboles = gr.Slider(10, 500, value=100, label="Nº de árboles")
```

Patrón muy recomendable: la interfaz principal queda limpia y los parámetros técnicos están disponibles para quien los necesite.

### Agrupación visual

```python
with gr.Group():        # junta los componentes sin separación visual
    gr.Textbox()
    gr.Textbox()
```

## 4.3 Ejemplo completo de estructura

```python
import gradio as gr

with gr.Blocks(title="Mi aplicación") as demo:
    gr.Markdown("# Título de la aplicación\nDescripción breve.")

    with gr.Tab("Predicción individual"):
        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown("### Entradas")
                x1 = gr.Slider(0, 10, label="Variable 1")
                x2 = gr.Slider(0, 10, label="Variable 2")
                with gr.Accordion("Avanzado", open=False):
                    umbral = gr.Slider(0, 1, value=0.5, label="Umbral")
                btn = gr.Button("Predecir", variant="primary")
            with gr.Column(scale=2):
                gr.Markdown("### Resultados")
                etiqueta = gr.Label()
                grafico  = gr.Plot()

    with gr.Tab("Lote desde CSV"):
        fichero = gr.File(file_types=[".csv"])
        tabla   = gr.Dataframe()

    btn.click(fn=predecir, inputs=[x1, x2, umbral],
              outputs=[etiqueta, grafico])

demo.launch()
```

---

# Módulo 5. El sistema de eventos

Los eventos son el mecanismo por el que la interfaz llama a tu código. Todos comparten la misma firma.

## 5.1 Firma común

```python
componente.evento(
    fn=funcion,                # qué ejecutar
    inputs=[comp1, comp2],     # de dónde salen los argumentos
    outputs=[comp3, comp4],    # dónde van los valores devueltos
)
```

`inputs` y `outputs` aceptan un componente suelto o una lista. Si la función no necesita argumentos, `inputs=None`.

## 5.2 Eventos disponibles

| Evento | Se dispara cuando... | Componentes típicos |
|---|---|---|
| `.click()` | se pulsa | `Button` |
| `.change()` | cambia el valor (por el usuario **o** por código) | `Slider`, `Textbox`, `Dropdown` |
| `.input()` | cambia el valor **solo** por acción del usuario | `Textbox`, `Slider` |
| `.submit()` | se pulsa Enter | `Textbox` |
| `.select()` | se selecciona un elemento | `Dataframe`, `Gallery`, `Radio` |
| `.upload()` | se sube un fichero | `File`, `Image`, `Audio` |
| `.clear()` | se limpia el componente | `Image`, `File` |
| `.blur()` | se pierde el foco | `Textbox` |
| `demo.load()` | se carga la página | el propio `Blocks` |

> **`.change()` frente a `.input()`.** La diferencia importa: si una función actualiza un componente que a su vez tiene un `.change()` conectado, se dispara en cascada y puedes provocar un bucle infinito. `.input()` solo reacciona a la interacción humana, por lo que es más seguro en interfaces reactivas complejas.

## 5.3 Reactividad en tiempo real

El patrón del cuaderno original: conectar `.change()` de cada slider para que la predicción se recalcule sin pulsar ningún botón.

```python
entradas = [sepal_l, sepal_w, petal_l, petal_w]
salidas  = [etiqueta, grafico, tabla]

btn.click(fn=predecir, inputs=entradas, outputs=salidas)

for slider in entradas:
    slider.change(fn=predecir, inputs=entradas, outputs=salidas)
```

## 5.4 Encadenar eventos

Los eventos devuelven un objeto que permite encadenar con `.then()`:

```python
btn.click(fn=validar, inputs=entrada, outputs=estado) \
   .then(fn=procesar, inputs=entrada, outputs=resultado) \
   .then(fn=graficar, inputs=resultado, outputs=grafico)
```

Cada paso espera a que termine el anterior. Útil para pipelines largos donde quieres ir mostrando progreso.

Variantes: `.success()` ejecuta el siguiente paso **solo si** el anterior no lanzó excepción.

## 5.5 Actualizar propiedades, no solo valores: `gr.update()`

Hasta ahora una función devuelve *valores*. A veces quieres cambiar el *aspecto* de un componente: ocultarlo, cambiar sus opciones, desactivarlo. Para eso está `gr.update()`:

```python
def cambiar_modo(modo):
    if modo == "Regresión":
        return gr.update(visible=True), gr.update(visible=False)
    else:
        return gr.update(visible=False), gr.update(visible=True)

selector.change(
    fn=cambiar_modo,
    inputs=selector,
    outputs=[panel_regresion, panel_clasificacion],
)
```

Otros usos habituales:

```python
gr.update(value=42)                          # cambiar el valor
gr.update(choices=["a", "b"], value="a")     # cambiar las opciones de un Dropdown
gr.update(interactive=False)                 # desactivar
gr.update(label="Nueva etiqueta")            # renombrar
```

> **Gradio 6:** se usa siempre `gr.update(...)`. La forma antigua `gr.Textbox.update(...)` de Gradio 3–4 ya no existe.

Un caso muy práctico: rellenar un desplegable con las columnas del CSV que el usuario acaba de subir.

```python
def leer_columnas(archivo):
    if archivo is None:
        return gr.update(choices=[], value=None)
    df = pd.read_csv(archivo.name)
    return gr.update(choices=list(df.columns), value=df.columns[0])

fichero.upload(fn=leer_columnas, inputs=fichero, outputs=selector_columna)
```

Esto resuelve una debilidad del ejemplo `analizar_csv` del cuaderno original, donde el usuario debe **escribir a mano** el nombre de la columna objetivo y recibe un error si se equivoca. Con este patrón, elige de una lista.

---

# Módulo 6. Estado, validación y funciones avanzadas

## 6.1 Estado por sesión: `gr.State`

Por defecto Gradio no recuerda nada entre llamadas. `gr.State` guarda un objeto Python **por usuario y por sesión**:

```python
with gr.Blocks() as demo:
    historial = gr.State([])          # valor inicial: lista vacía

    entrada = gr.Textbox()
    salida  = gr.JSON()
    btn     = gr.Button("Añadir")

    def añadir(texto, hist):
        hist = hist + [texto]         # crear lista nueva, no mutar
        return hist, hist

    btn.click(fn=añadir,
              inputs=[entrada, historial],
              outputs=[historial, salida])
```

El estado se pasa como argumento y se devuelve actualizado. Dos avisos:

- **No lo uses como base de datos.** Se pierde al recargar la página.
- **Crea objetos nuevos en lugar de mutar los existentes.** Mutar puede provocar comportamientos difíciles de depurar cuando hay varios usuarios.

## 6.2 Mensajes al usuario: `gr.Error`, `gr.Warning`, `gr.Info`

Si tu función lanza una excepción normal, el usuario ve un error genérico y feo. Gradio ofrece tres niveles:

```python
def procesar(archivo, columna):
    if archivo is None:
        raise gr.Error("Debes subir un fichero CSV.")        # rojo, detiene

    df = pd.read_csv(archivo.name)

    if columna not in df.columns:
        raise gr.Error(f"La columna '{columna}' no existe. "
                       f"Disponibles: {', '.join(df.columns)}")

    if df.isna().sum().sum() > 0:
        gr.Warning("El fichero tiene valores nulos; se imputarán con la mediana.")

    gr.Info(f"Procesando {len(df)} filas...")                # informativo
    return analizar(df)
```

`gr.Error` interrumpe la ejecución; `gr.Warning` y `gr.Info` solo muestran un aviso y continúan. Esta es una mejora directa sobre el patrón del cuaderno, que devuelve los errores como texto en la salida.


## 6.3 Barras de progreso

Para funciones lentas, dar retroalimentación evita que el usuario piense que la aplicación se ha colgado:

```python
def entrenar(n_modelos, progress=gr.Progress()):
    resultados = []
    for i in progress.tqdm(range(n_modelos), desc="Entrenando"):
        resultados.append(entrenar_uno(i))
    return resultados
```

Basta con declarar el parámetro `progress=gr.Progress()`; Gradio lo inyecta solo, no aparece en la interfaz. También admite control manual:

```python
def procesar(datos, progress=gr.Progress()):
    progress(0, desc="Cargando")
    ...
    progress(0.5, desc="Entrenando")
    ...
    progress(1.0, desc="Listo")
```

## 6.4 Salida progresiva con generadores

Si tu función es un generador (usa `yield` en vez de `return`), Gradio muestra cada resultado intermedio conforme se produce:

```python
def generar_texto(prompt):
    texto = ""
    for palabra in modelo_llm.stream(prompt):
        texto += palabra
        yield texto        # la interfaz se actualiza en cada iteración
```

Es el mecanismo detrás de todos los chats de LLM que has visto hechos con Gradio.

## 6.5 Ejemplos precalculados

```python
gr.Examples(
    examples=[[5.1, 3.5, 1.4, 0.2], [6.7, 3.1, 4.7, 1.5]],
    inputs=[sl, sw, pl, pw],
    outputs=[etiqueta, grafico],
    fn=predecir,
    cache_examples=True,      # calcula al arrancar, respuesta instantánea
)
```

Con `cache_examples=True` los resultados se calculan una vez al arrancar y se guardan. El usuario que pulsa un ejemplo ve el resultado al instante. Muy recomendable si el modelo es lento.

## 6.6 Temas y estilo

```python
demo.launch(theme=gr.themes.Soft())
```

Temas incluidos: `Default()`, `Soft()`, `Glass()`, `Monochrome()`, `Base()`, `Origin()`, `Citrus()`, `Ocean()`.

> **Cambio de Gradio 6:** el parámetro `theme` **se ha movido** del constructor `gr.Blocks(theme=...)` al método `launch(theme=...)`. El código del cuaderno original usa la forma antigua y en Gradio 6 emite el aviso:
>
> ```
> UserWarning: The parameters have been moved from the Blocks constructor
> to the launch() method in Gradio 6.0: theme.
> ```
>
> Sigue funcionando por retrocompatibilidad, pero conviene actualizarlo.

Personalización con CSS:

```python
css_propio = """
#mi-boton { background: #2563eb; font-weight: 600; }
.gradio-container { max-width: 1100px !important; }
"""

with gr.Blocks(css=css_propio) as demo:
    btn = gr.Button("Enviar", elem_id="mi-boton")
```

---

# Módulo 7. Del cuaderno a la aplicación

Aquí está el salto conceptual más importante del curso. Una aplicación que vive en un cuaderno **no es desplegable**. Hay que reorganizarla.

## 7.1 Los tres problemas del código en el cuaderno

Tomando como referencia la sección 4.2 del cuaderno original:

**Problema 1: el modelo se entrena al arrancar.**

```python
clf_iris = RandomForestClassifier(n_estimators=100, random_state=42)
clf_iris.fit(iris.data, iris.target)
```

Con Iris esto tarda milisegundos, pero con un modelo real puede tardar horas. Un servidor que reentrena en cada arranque es inviable. **Solución:** separar entrenamiento de servicio, y serializar el modelo.

**Problema 2: el estado está disperso por el cuaderno.**

La función `predecir_iris` usa `clf_iris`, `nombres_iris` e `iris`, definidos en celdas anteriores. En un cuaderno funciona por el estado global del kernel; en un fichero `.py` hay que hacerlo explícito.

**Problema 3: `share=True` no es un despliegue.**

El enlace de `share=True` **caduca a las 72 horas** y solo funciona mientras la celda de Colab siga ejecutándose. Sirve para enseñar algo a un compañero esta tarde, no para publicar.

## 7.2 La estructura correcta

```
mi_aplicacion/
├── app.py                  # la aplicación (esto es lo que se ejecuta)
├── entrenar_modelo.py      # entrena y serializa (NO se ejecuta en el servidor)
├── modelo_iris.joblib      # el modelo ya entrenado
├── requirements.txt        # dependencias
└── README.md               # documentación + configuración del Space
```

## 7.3 Paso 1: serializar el modelo

```python
# entrenar_modelo.py
import joblib
import sklearn
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

iris = load_iris()
X, y = iris.data, iris.target

modelo = RandomForestClassifier(n_estimators=100, random_state=42)

scores = cross_val_score(modelo, X, y, cv=5)
print(f"Accuracy validación cruzada: {scores.mean():.4f}")

modelo.fit(X, y)

# Guardamos el modelo Y los metadatos que la app necesitará
artefacto = {
    "modelo": modelo,
    "nombres_clases": [str(c) for c in iris.target_names],
    "nombres_features": [str(f) for f in iris.feature_names],
    "sklearn_version": sklearn.__version__,
}

joblib.dump(artefacto, "modelo_iris.joblib")
```

Dos detalles que evitan problemas reales:

- **Validación cruzada en lugar de evaluar sobre los datos de entrenamiento.** El cuaderno original calcula la accuracy del modelo Iris sobre los mismos datos con los que se entrenó, lo que da un 1.0000 engañoso. Con validación cruzada sale 0.9667, que es la cifra honesta.
- **Guardar los metadatos junto al modelo.** Así `app.py` no necesita importar `load_iris` ni depender de scikit-learn para nada más que cargar el modelo.

## 7.4 Paso 2: escribir `app.py`

```python
import os
import gradio as gr
import joblib
import matplotlib
matplotlib.use("Agg")          # backend sin ventana: OBLIGATORIO en servidor
import matplotlib.pyplot as plt
import numpy as np

# Carga UNA vez, al arrancar, fuera de cualquier función
RUTA = os.path.join(os.path.dirname(__file__), "modelo_iris.joblib")
artefacto = joblib.load(RUTA)
modelo = artefacto["modelo"]
NOMBRES_CLASES = artefacto["nombres_clases"]

def predecir(sl, sw, pl, pw):
    if any(v is None for v in (sl, sw, pl, pw)):
        raise gr.Error("Faltan medidas por introducir.")
    probs = modelo.predict_proba(np.array([[sl, sw, pl, pw]]))[0]
    return {c: float(p) for c, p in zip(NOMBRES_CLASES, probs)}

with gr.Blocks(title="Clasificador Iris") as demo:
    ...

if __name__ == "__main__":
    demo.launch(theme=gr.themes.Soft())
```

Tres reglas que se ven aquí:

1. **`matplotlib.use("Agg")` antes de importar `pyplot`.** Sin esto, matplotlib intenta abrir una ventana gráfica y la aplicación falla en un servidor sin pantalla.
2. **La carga del modelo va fuera de la función.** Dentro se recargaría en cada clic.
3. **`if __name__ == "__main__":`** protege el `launch()`, para que importar el módulo (por ejemplo, en un test) no arranque un servidor.

## 7.5 Paso 3: `requirements.txt`

```
gradio==6.20.0
scikit-learn==1.8.0
joblib>=1.4
numpy>=2.0
pandas>=2.2
matplotlib>=3.8
```

> **Regla crítica:** la versión de scikit-learn debe ser **la misma** con la que generaste el `.joblib`. Cargar un modelo serializado con una versión distinta produce avisos en el mejor caso y errores o predicciones incorrectas en el peor. Por eso `entrenar_modelo.py` imprime la versión que ha usado.

Consejo adicional: **no incluyas paquetes que no uses.** Cada línea de `requirements.txt` es tiempo de arranque. Si tu aplicación no dibuja con seaborn, no lo pongas.

---

# Módulo 8. Despliegue permanente en Hugging Face Spaces

Este es el módulo que responde a la pregunta de fondo: **cómo publicar la aplicación para que funcione online sin depender del cuaderno**.

## 8.1 Por qué Hugging Face Spaces

Un Space es, técnicamente, **un repositorio de Git que además ejecuta código**. Cuando subes un `app.py` y un `requirements.txt`, la plataforma construye un contenedor, instala las dependencias y arranca tu aplicación en una URL permanente del tipo:

```
https://TU_USUARIO-nombre-del-space.hf.space
```

Ventajas frente a `share=True`:

| | `share=True` | Hugging Face Spaces |
|---|---|---|
| Duración | 72 horas | Permanente |
| Requiere el cuaderno abierto | Sí | No |
| URL | Aleatoria, cambia cada vez | Fija |
| Coste | Gratis | Gratis (CPU Basic) |
| Control de versiones | No | Git |

La plataforma ofrece 16 GB de RAM, 2 núcleos de CPU y 50 GB de disco no persistente de forma gratuita en el nivel CPU Basic. Es más que suficiente para modelos clásicos de scikit-learn.

## 8.2 Requisitos previos

1. Una cuenta gratuita en [huggingface.co](https://huggingface.co).
2. Los cuatro ficheros del módulo 7: `app.py`, `modelo_iris.joblib`, `requirements.txt`, `README.md`.


## 8.3 El fichero `README.md` y su cabecera YAML

Este es el detalle que se le escapa a todo el mundo la primera vez. **El `README.md` de un Space no es solo documentación: su cabecera YAML configura el Space.**

```markdown
---
title: Clasificador Iris
emoji: 🌸
colorFrom: blue
colorTo: green
sdk: gradio
sdk_version: 6.20.0
app_file: app.py
pinned: false
license: mit
---

# Clasificador de flores Iris

Texto normal de documentación a partir de aquí...
```

Campos importantes:

| Campo | Qué hace |
|---|---|
| `sdk` | `gradio`, `docker` o `static`. Para nosotros, `gradio`. |
| `sdk_version` | Versión de Gradio a instalar. **Debe coincidir con la de `requirements.txt`.** |
| `app_file` | El fichero de entrada. Por convención `app.py`. |
| `title`, `emoji`, `colorFrom`, `colorTo` | Aspecto de la tarjeta en el listado de Spaces. |
| `license` | Licencia del repositorio. |

Los tres guiones de apertura y cierre son obligatorios y deben estar en las primeras líneas del fichero, sin nada antes.

## 8.4 Método A: subida por navegador (el más sencillo)

Recomendado para empezar, no requiere Git.

**Paso 1.** Ve a [huggingface.co/spaces](https://huggingface.co/spaces) y pulsa **Create new Space**.

**Paso 2.** Rellena el formulario:
- *Space name*: `clasificador-iris` (será parte de la URL).
- *License*: MIT, por ejemplo.
- *Select the Space SDK*: **Gradio**.
- *Space hardware*: **CPU basic — FREE**.
- *Visibility*: **Public**.

**Paso 3.** Pulsa **Create Space**. Verás un repositorio vacío con instrucciones.

**Paso 4.** Ve a la pestaña **Files** → **Add file** → **Upload files**. Arrastra los cuatro ficheros:

```
app.py
modelo_iris.joblib
requirements.txt
README.md
```

> Cuidado: al crear el Space, HF genera automáticamente un `README.md` con la cabecera YAML. Si subes el tuyo, lo sobrescribes. Asegúrate de que el tuyo lleva la cabecera YAML completa, o edita el suyo en lugar de reemplazarlo.

**Paso 5.** Escribe un mensaje de commit y pulsa **Commit changes to main**.

**Paso 6.** Ve a la pestaña **App**. Verás el estado **Building** mientras instala las dependencias (2–5 minutos la primera vez). Cuando pase a **Running**, tu aplicación está online.

Si aparece **Build error** o **Runtime error**, pulsa en **Logs** para ver el traceback completo. La sección 8.8 recoge los fallos más habituales.

## 8.5 Método B: por Git (recomendado para trabajo continuado)

Permite versionar, trabajar en local y desplegar con un `git push`.

```bash
# 1. Instalar el cliente y autenticarse
pip install huggingface_hub
hf auth login          # pega un token de https://huggingface.co/settings/tokens
                       # el token debe tener permiso de escritura (write)

# 2. Clonar el Space que has creado en la web
git clone https://huggingface.co/spaces/TU_USUARIO/clasificador-iris
cd clasificador-iris

# 3. Copiar tus ficheros a la carpeta
cp ../app.py ../requirements.txt ../modelo_iris.joblib .

# 4. Ficheros grandes: Git LFS
#    Cualquier fichero de más de 10 MB debe ir por LFS
git lfs install
git lfs track "*.joblib"
git lfs track "*.pkl"
git lfs track "*.h5"
git lfs track "*.pt"
git add .gitattributes

# 5. Publicar
git add .
git commit -m "Primera versión de la aplicación"
git push
```

Cada `git push` dispara automáticamente una reconstrucción del Space. Este es el flujo de trabajo real: desarrollas en local con `python app.py`, y cuando funciona, haces push.

## 8.6 Crear el Space directamente desde Colab

Si prefieres no salir del cuaderno, se puede hacer todo por código:

```python
!pip install huggingface_hub --quiet

from huggingface_hub import HfApi, login

# Token con permiso de escritura: https://huggingface.co/settings/tokens
login(token="hf_xxxxxxxxxxxxx")

api = HfApi()
REPO = "TU_USUARIO/clasificador-iris"

# 1. Crear el Space (solo la primera vez)
api.create_repo(
    repo_id=REPO,
    repo_type="space",
    space_sdk="gradio",
    exist_ok=True,
)

# 2. Subir la carpeta entera
api.upload_folder(
    folder_path="app_iris_gradio",   # carpeta local con los 4 ficheros
    repo_id=REPO,
    repo_type="space",
)

print(f"Desplegado en: https://huggingface.co/spaces/{REPO}")
```

En Colab, guarda el token en los **Secrets** (icono de la llave) en lugar de escribirlo en una celda:

```python
from google.colab import userdata
login(token=userdata.get("HF_TOKEN"))
```

## 8.7 Configuración adicional del Space


### Secretos y variables de entorno

Si tu aplicación necesita una clave de API, **nunca la escribas en `app.py`** (el código es público y HF tiene un escáner que detecta credenciales). Ve a **Settings** → **Variables and secrets**:

- **Variables**: valores de configuración no sensibles. Son públicos.
- **Secrets**: claves y tokens. Privados, no se pueden volver a leer una vez guardados.

Ambos llegan a tu aplicación como variables de entorno:

```python
import os
clave = os.getenv("MI_API_KEY")
```

### Hibernación

En hardware gratuito, un Space **se duerme tras un periodo de inactividad** (actualmente 48 horas). No se pierde nada: al recibir la siguiente visita se reinicia solo, aunque el primer usuario esperará unos segundos. Para evitarlo hace falta hardware de pago.

### Visibilidad

- **Public**: código y aplicación visibles para todos.
- **Private**: solo tú. La aplicación no es accesible desde fuera.
- **Protected** (planes de pago): el código queda privado pero la aplicación sigue siendo accesible por URL. Útil si quieres publicar la demo sin publicar el código.


### Variables de entorno que Spaces proporciona

Tu código puede detectar si está corriendo en Spaces:

```python
import os

EN_SPACES = os.getenv("SPACE_ID") is not None

if EN_SPACES:
    demo.launch()                    # la URL ya es pública
else:
    demo.launch(share=False, debug=True)   # desarrollo en local
```

## 8.8 Errores frecuentes en el despliegue

| Síntoma | Causa habitual | Solución |
|---|---|---|
| `Build error` nada más empezar | Falta `requirements.txt` o tiene un nombre de paquete mal escrito | Revisa los logs; el paquete es `scikit-learn`, no `sklearn` |
| `Could not find a version that satisfies the requirement X` | La versión que pediste **no existe para el Python del Space** (por defecto 3.10) | Ver sección 8.9 |
| `Ignored the following versions that require a different python version` | Lo mismo que el anterior. Es la pista definitiva. | Ver sección 8.9 |
| `ModuleNotFoundError` | La dependencia no está en `requirements.txt` | Añádela y haz commit |
| `FileNotFoundError: modelo.joblib` | El modelo no se subió (o Git LFS no estaba configurado) | Comprueba en la pestaña Files que el fichero está y pesa lo correcto |
| `InconsistentVersionWarning` de sklearn | El `.joblib` se generó con otra versión | Fija la versión exacta en `requirements.txt` |
| Aplicación en blanco, sin errores | Falta `app_file` en el YAML o el fichero no se llama `app.py` | Corrige la cabecera del README |
| Falla al dibujar gráficos | matplotlib intenta abrir una ventana | Añade `matplotlib.use("Agg")` antes de importar `pyplot` |
| Arranca y se para: `Stopping Node.js server...` | El modo SSR de Gradio 6 | `demo.launch(ssr_mode=False)`. Ver sección 8.10 |
| Interfaz en blanco o elementos no clicables | Fallos de hidratación del SSR | Igual: `ssr_mode=False` |
| El Space se queda en `Building` mucho tiempo | Dependencias muy pesadas (torch, tensorflow) | Normal; usa versiones CPU (`torch --index-url .../cpu`) |
| `Runtime error` con modelos grandes | Se supera la RAM disponible | Reduce el modelo o cuantízalo |

**Cómo depurar en serio.** La pestaña **Logs** del Space muestra tanto el log de construcción (`Build logs`) como el de ejecución (`Container logs`). El 90 % de los problemas se resuelven leyendo ahí el traceback. Antes de subir nada, prueba siempre en local:

```bash
pip install -r requirements.txt
python app.py
```

Si funciona en local con un entorno limpio, funcionará en Spaces.


## 8.9 Caso práctico: incompatibilidad entre versión de paquete y versión de Python

Este error merece su propia sección porque es el más desconcertante de los habituales. Un ejemplo real al desplegar la aplicación de este curso:

```
ERROR: Ignored the following versions that require a different python version:
       1.8.0 Requires-Python >=3.11; 1.9.0 Requires-Python >=3.11
ERROR: Could not find a version that satisfies the requirement scikit-learn==1.8.0
       (from versions: 0.9, 0.10, ..., 1.7.0, 1.7.1, 1.7.2)
ERROR: No matching distribution found for scikit-learn==1.8.0
```

**Cómo leerlo.** La primera línea es la clave, y es fácil pasarla por alto porque la segunda parece el error principal. Dice literalmente que existe una versión 1.8.0, pero que **requiere Python 3.11 o superior**. La lista de "versions" que ofrece termina en 1.7.2: esas son las que sí son compatibles con el Python que está usando el Space.

**Por qué ocurre.** Los Spaces usan **Python 3.10 por defecto**. Si entrenaste el modelo en Colab o en tu ordenador con un Python más moderno, es probable que tengas instalada una versión de scikit-learn que ya no publica paquetes para 3.10. El fichero `.joblib` se generó con esa versión, y al fijarla en `requirements.txt` el Space no puede instalarla.

Es la manifestación concreta de una regla más general: **el entorno donde entrenas y el entorno donde despliegas tienen que ser compatibles.** No basta con fijar la versión de la librería; hay que fijar también la del intérprete.


### Solución A: subir la versión de Python del Space

Añade una línea a la cabecera YAML del `README.md`:

```yaml
---
title: Clasificador Iris
emoji: 🌸
colorFrom: blue
colorTo: green
sdk: gradio
sdk_version: 6.20.0
python_version: "3.11"
app_file: app.py
pinned: false
license: mit
---
```

El campo `python_version` acepta cualquier versión válida de Python 3.x y su valor por defecto es 3.10. Las comillas no son obligatorias, pero evitan que YAML interprete `3.10` como el número 3.1.

Ventaja: no tocas el modelo ni el resto de ficheros. Es un cambio de una línea.

### Solución B: bajar la versión de la librería y regenerar el modelo

Si prefieres no salir de la configuración por defecto, usa la última versión compatible con Python 3.10:

```
gradio==6.20.0
scikit-learn==1.7.2
joblib>=1.4
numpy>=1.26
pandas>=2.2
matplotlib>=3.8
```

**Importante:** no basta con cambiar el `requirements.txt`. Hay que **regenerar el `.joblib`** con esa misma versión, o volverás al problema de incompatibilidad entre el modelo serializado y la librería que lo carga:

```bash
pip install scikit-learn==1.7.2
python entrenar_modelo.py          # produce un modelo_iris.joblib nuevo
```

Luego sube el `.joblib` regenerado junto con el `requirements.txt` corregido.

### Cuál elegir

| | Solución A (Python 3.11) | Solución B (scikit-learn 1.7.2) |
|---|---|---|
| Esfuerzo | Una línea en el README | Reinstalar, reentrenar, resubir |
| Toca el modelo | No | Sí |
| Riesgo | Bajo; Gradio soporta 3.10–3.13 | Bajo; 1.7.2 es estable |
| Recomendada si... | acabas de encontrarte el error | quieres máxima compatibilidad con el entorno estándar de Spaces |

En la práctica, **empieza por la A**. Si algún otro paquete de tu proyecto no tuviera soporte para 3.11, entonces recurre a la B.

### Cómo evitarlo desde el principio

Antes de fijar una versión en `requirements.txt`, comprueba con qué Python es compatible:

```bash
pip index versions scikit-learn
```

O consulta la página del paquete en PyPI, en el apartado *Requires: Python*. Un hábito que ahorra mucho tiempo: **entrena el modelo en un entorno con la misma versión de Python que vayas a desplegar.**

## 8.10 Caso práctico: la aplicación arranca y se detiene sola

Otro error real, distinto del anterior porque **no es de construcción sino de ejecución**. El Space compila bien, y en los logs aparece:

```
===== Application Startup at 2026-07-22 07:43:55 =====

* Running on local URL:  http://0.0.0.0:7860, with SSR (Node proxy -> Python :7861)
* To create a public link, set `share=True` in `launch()`.

Stopping Node.js server...
```

La aplicación arranca correctamente (fíjate en que anuncia la URL) y acto seguido se apaga. No hay traceback, no hay excepción: simplemente termina.

**La pista está en `with SSR (Node proxy -> Python :7861)`.**

Desde Gradio 5, el framework activa por defecto el **server-side rendering (SSR)** cuando detecta que corre en Spaces. En ese modo, Gradio no sirve la página solo desde Python: arranca **un segundo servidor, en Node.js**, que hace de proxy delante del servidor de Python. El objetivo es que la primera carga de la página sea más rápida.

El problema es que ese servidor de Node es una pieza frágil dentro del contenedor de Spaces. Cuando falla o no consigue mantenerse, arrastra consigo al proceso principal, y el mensaje que queda en el log es precisamente `Stopping Node.js server...`. Es un problema conocido y recurrente del proyecto, con varias incidencias abiertas.

### La solución

Desactivar el SSR en la llamada a `launch()`:

```python
if __name__ == "__main__":
    demo.launch(
        theme=gr.themes.Soft(),
        ssr_mode=False,
        server_name="0.0.0.0",
    )
```

Lo único que pierdes es una carga inicial marginalmente más lenta. A cambio, la aplicación se comporta igual en local que en el servidor, que es lo que interesa.

`server_name="0.0.0.0"` no está relacionado con el SSR, pero conviene ponerlo: hace que el servidor escuche en todas las interfaces de red. Sin eso, en un contenedor la aplicación solo se escucharía a sí misma. En Spaces suele funcionar sin declararlo porque la plataforma define la variable de entorno correspondiente, pero al ser explícito el mismo `app.py` sirve también para Docker.


### Síntomas relacionados del mismo origen

El SSR provoca más problemas que este, y todos se resuelven igual:

- La interfaz se ve **en blanco** o a medio dibujar.
- Los botones **no responden** al clic.
- Las **fuentes** no se cargan y el texto sale con una tipografía por defecto.
- Errores **500** intermitentes al abrir el Space.

Si ves cualquiera de estos, prueba `ssr_mode=False` antes de investigar nada más.

### Cómo distinguir este error del anterior

Merece la pena tener clara la diferencia, porque el sitio donde hay que mirar no es el mismo:

| | Error de construcción (8.9) | Error de ejecución (8.10) |
|---|---|---|
| Cuándo ocurre | Instalando dependencias | Después de `Application Startup` |
| Dónde se ve | `Build logs` | `Container logs` |
| Qué se toca | `requirements.txt` / `README.md` | `app.py` |
| Ejemplo | `No matching distribution found` | `Stopping Node.js server...` |

La línea `===== Application Startup at ... =====` es el separador: todo lo anterior es construcción, todo lo posterior es ejecución.


---

# Módulo 9. Alternativas de despliegue

Hugging Face Spaces no es la única opción, aunque para modelos de ML es la más cómoda.

## 9.1 Comparativa

| Plataforma | Gratis | Complejidad | Cuándo elegirla |
|---|---|---|---|
| **HF Spaces** | Sí (CPU Basic) | Muy baja | Demos de ML. Opción por defecto. |
| **Render** | Capa gratuita limitada | Baja | Si ya usas Render para otras cosas |
| **Railway** | Crédito mensual | Baja | Despliegues con base de datos |
| **Google Cloud Run** | Capa gratuita | Media | Producción real, escalado automático |
| **Docker + VPS** | No | Alta | Control total, datos sensibles |

## 9.2 Despliegue con Docker

Si necesitas portabilidad total, un `Dockerfile` mínimo para Gradio:

```dockerfile
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 7860

# 0.0.0.0 es imprescindible: sin esto la app solo escucha
# dentro del contenedor y no es accesible desde fuera
ENV GRADIO_SERVER_NAME=0.0.0.0
ENV GRADIO_SERVER_PORT=7860

CMD ["python", "app.py"]
```

Construir y ejecutar:

```bash
docker build -t mi-app-gradio .
docker run -p 7860:7860 mi-app-gradio
```

El mismo `Dockerfile` sirve para un Space de tipo `docker` (cambiando `sdk: docker` en el README) o para cualquier VPS.

## 9.3 Integrar Gradio en una API de FastAPI

Cuando la aplicación es solo una parte de un sistema mayor:

```python
from fastapi import FastAPI
import gradio as gr

app = FastAPI()

@app.get("/api/salud")
def salud():
    return {"estado": "ok"}

# Monta la interfaz de Gradio en una ruta concreta
app = gr.mount_gradio_app(app, demo, path="/demo")

# uvicorn main:app --host 0.0.0.0 --port 8000
```

Así tienes la API REST en `/api/...` y la interfaz visual en `/demo`.

---

# Módulo 10. Buenas prácticas y errores frecuentes

## 10.1 Estructura del código

**Separa la lógica de la interfaz.** Tus funciones de predicción no deberían saber que existe Gradio (más allá de `gr.Error`). Así puedes testearlas:

```python
# logica.py
def predecir(medidas: list[float]) -> dict[str, float]:
    ...

# test_logica.py
def test_predecir_setosa():
    r = predecir([5.1, 3.5, 1.4, 0.2])
    assert max(r, key=r.get) == "setosa"

# app.py
from logica import predecir
```

**Carga los recursos pesados una sola vez**, a nivel de módulo, nunca dentro de la función de predicción.

**Usa `if __name__ == "__main__":`** para proteger el `launch()`.


## 10.2 Rendimiento

- **Cachea los ejemplos** con `cache_examples=True` si el modelo es lento.
- **Limita la concurrencia** si el modelo consume mucha memoria: `demo.queue(default_concurrency_limit=2)`.
- **Activa la cola** para funciones lentas: `demo.queue()` antes de `launch()`. Evita que las peticiones simultáneas se pisen.
- **No devuelvas objetos gigantes**: un DataFrame de un millón de filas bloqueará el navegador.

## 10.3 Seguridad

- **Nunca subas credenciales al repositorio.** Usa los secretos del Space.
- **Valida siempre la entrada del usuario.** Un `gr.File` acepta cualquier cosa que el usuario suba.
- **Cuidado con `gr.Code` o cualquier ejecución dinámica.** No uses `eval()` sobre entradas del usuario.
- **Protege con contraseña si hace falta**: `demo.launch(auth=("usuario", "contraseña"))`. Para algo más serio, `auth` acepta una función de verificación.

## 10.4 Experiencia de usuario

- Pon `label` e `info` en **todos** los componentes.
- Añade `gr.Examples`: reduce drásticamente la fricción del primer uso.
- Usa `demo.load()` para mostrar un resultado inicial en lugar de una pantalla vacía.
- Explica las **limitaciones del modelo** en un `gr.Markdown`. Si tu clasificador se entrenó con 150 flores, dilo.
- Usa `gr.Error` con mensajes accionables: no "Error", sino "La columna 'x' no existe. Disponibles: a, b, c".


## 10.5 Los cinco errores que más tiempo cuestan

1. **Descuadre entre entradas, argumentos y salidas.** Si `outputs` tiene 3 componentes, la función debe devolver exactamente 3 valores.
2. **Olvidar `matplotlib.use("Agg")`.** Funciona en Colab, falla en el servidor.
3. **Tipos de NumPy en las salidas.** `np.float32` no es serializable a JSON. Convierte con `float()`.
4. **No fijar versiones.** El código funciona hoy y se rompe en el siguiente despliegue.
5. **Entrenar el modelo dentro de `app.py`.** Funciona con Iris, es inviable con cualquier cosa real.

---

# Apéndice A. Cambios de Gradio 5 a Gradio 6

Gradio 6 (noviembre de 2025) introdujo cambios que rompen código anterior. Si sigues tutoriales o cuadernos de 2024–2025, estos son los que más te afectarán:

| Cambio | Gradio 5 | Gradio 6 |
|---|---|---|
| **Tema** | `gr.Blocks(theme=...)` | `demo.launch(theme=...)` |
| Actualizar componentes | `gr.Textbox.update(...)` (v3-4) | `gr.update(...)` |
| Visibilidad de la API | `show_api=False` en eventos | `api_visibility="hidden"` / `"undocumented"` |
| `gr.HTML` | `padding=True` por defecto | `padding=False` por defecto |
| `gr.Chatbot` | `allow_tags=False` por defecto | `allow_tags=True` por defecto |
| `gr.Dataframe` | `row_count` / `col_count` | reestructurados |
| `AppError` (cliente) | subclase de `ValueError` | subclase de `Exception` |

El más visible al ejecutar el cuaderno original es el del tema:

```
UserWarning: The parameters have been moved from the Blocks constructor
to the launch() method in Gradio 6.0: theme.
```

Corrección:

```python
# Antes
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    ...
demo.launch()

# Ahora
with gr.Blocks() as demo:
    ...
demo.launch(theme=gr.themes.Soft())
```

Guía oficial completa: `gradio.app/guides/gradio-6-migration-guide`.

> **Recomendación práctica.** Gradio evoluciona rápido y ha tenido varias versiones con regresiones. Para material docente o cualquier cosa que deba seguir funcionando dentro de seis meses, **fija la versión exacta** en `requirements.txt` y en `sdk_version`. Actualiza solo cuando tengas tiempo de probar.

---

# Apéndice B. Chuleta de referencia rápida

## Esqueleto mínimo

```python
import gradio as gr

def fn(x):
    return x

with gr.Blocks() as demo:
    inp = gr.Textbox()
    out = gr.Textbox()
    btn = gr.Button("Ir")
    btn.click(fn, inputs=inp, outputs=out)

if __name__ == "__main__":
    demo.launch()
```

## Componentes por tipo de dato

```python
gr.Textbox()      -> str
gr.Number()       -> float
gr.Slider()       -> float
gr.Checkbox()     -> bool
gr.Radio()        -> str
gr.Dropdown()     -> str | list[str]
gr.File()         -> objeto con .name (ruta)
gr.Image()        -> np.ndarray  (o PIL / ruta según type=)
gr.Dataframe()    -> pd.DataFrame
gr.Label()        <- dict[str, float]
gr.Plot()         <- figura matplotlib / plotly
gr.JSON()         <- dict | list
```

## Layout

```python
with gr.Row():          ...   # horizontal
with gr.Column(scale=2): ...  # vertical, ancho proporcional
with gr.Tab("Nombre"):  ...   # pestañas
with gr.Accordion("Título", open=False): ...  # plegable
with gr.Group():        ...   # agrupación visual
```

## Eventos

```python
btn.click(fn, inputs, outputs)
slider.change(fn, inputs, outputs)     # cambio por usuario O código
caja.input(fn, inputs, outputs)        # solo por usuario
caja.submit(fn, inputs, outputs)       # Enter
fichero.upload(fn, inputs, outputs)
demo.load(fn, inputs, outputs)         # al cargar la página

btn.click(a, ...).then(b, ...)         # encadenar
btn.click(a, ...).success(b, ...)      # solo si a no falla
```

## Actualizar propiedades

```python
gr.update(value=1)
gr.update(visible=False)
gr.update(interactive=False)
gr.update(choices=["a", "b"], value="a")
gr.update(label="Nuevo")
```

## Mensajes

```python
raise gr.Error("Mensaje")   # rojo, detiene la ejecución
gr.Warning("Mensaje")       # naranja, continúa
gr.Info("Mensaje")          # azul, continúa
```

## Lanzamiento

```python
demo.launch(
    share=False,             # True -> enlace público temporal (72 h)
    server_name="0.0.0.0",   # necesario en Docker
    server_port=7860,
    auth=("usuario", "clave"),
    theme=gr.themes.Soft(),
    debug=True,
    show_error=True,
    pwa=True,                # instalable como aplicación
    ssr_mode=False,          # imprescindible en Hugging Face Spaces
)

demo.queue(default_concurrency_limit=2)   # antes de launch()
```

## Cabecera del README para Spaces

```yaml
---
title: Mi aplicación
emoji: 🚀
colorFrom: blue
colorTo: green
sdk: gradio
sdk_version: 6.20.0
app_file: app.py
pinned: false
license: mit
---
```

---

## Recursos

- Documentación oficial: `gradio.app/docs`
- Guías: `gradio.app/guides`
- Playground interactivo: `gradio.app/playground`
- Documentación de Spaces: `huggingface.co/docs/hub/spaces`
- Guía de migración a Gradio 6: `gradio.app/guides/gradio-6-migration-guide`
